In [ ]:
import pandas as pd
from pathlib import Path


In [ ]:
project_root = Path.cwd()
if not Path("data").exists():
    project_root = project_root.parent          

print(project_root)

In [ ]:
data_file = Path(project_root/ "data/raw/cu_income.csv")
data_file.exists()

In [ ]:
ce_data = pd.read_csv(data_file, header=None)

In [ ]:
ce_data.head(10)

In [ ]:
ce_selected = ce_data.iloc[[3,4,5, 9,51], 0:7]
ce_selected.head()

In [ ]:
# set column names and row index 
ce = ce_selected.copy()

ce.columns = ce.iloc[0]
ce = ce.iloc[1:]
ce.columns.name = None

ce = ce.set_index(ce.columns[0])
ce.index.name = None

ce

In [ ]:
ce.columns

In [ ]:
ce.index

In [ ]:
# rename columns and rows
ce = ce.rename(columns={
    "Lowest\n20\npercent": "lowest20pct",
    "Second\n20\npercent": "second20pct",
    "Third\n20\npercent": "third20pct",
    "Fourth\n20\npercent": "fourth20pct",
    "Highest\n20\npercent": "highest20pct"
})

ce = ce.rename(index={
    'Number of consumer units (in thousands) a/ ': 'num_cu',
    'Percent distribution of consumer units': 'pct_cu',
    'Income before taxes': 'income_mean',
    'Food at home': 'grocery'
})
ce

## Calculate two statistics: group grocery expenditure, unit grocery expenditure
### group grocery expenditure = aggregated grocery expenditure * expenditure share / 100
### average grocery expenditure per consumer unit = 1000 * group grocery expenditure / number of consumer unites in each group 

In [ ]:
ce = (
    ce.replace(r"[$,]", "", regex=True)    # regular expression 
    .apply(pd.to_numeric, errors="coerce")
)
ce.dtypes

In [ ]:
ce_t = ce.T
ce_t['grocery_pct'] = ce_t['grocery']
ce_t.loc['Aggregate','grocery_pct'] = 100
ce_t

In [ ]:
ce_t["grocery_group"] = ce_t.loc["Aggregate","grocery"] * ce_t["grocery_pct"] / 100
ce_t["avg_grocery"] = 1000 * ce_t["grocery_group"] / ce_t["num_cu"]

ce_t

### Insert a column of the lowest average income for each income group for reference
### These numbers are directly from Bureau of Labor Statistics based on nationwide population. 

In [ ]:
ce_t["highest_income"] = [9999999, 29932, 57452, 94511, 155925, 9999999]
ce_t